In [816]:
import pandas as pd
import gensim
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Dense, Dropout, Embedding, LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
import numpy as np
from sklearn.model_selection import train_test_split
from nltk.stem import WordNetLemmatizer
import re
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [817]:
df = pd.read_csv('spamhamdata.csv',sep='\t', header=None, names=['class', 'text'])
df


,class,text
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives around here though"
...,...,...
5567,spam,"This is the 2nd time we have tried 2 contact u. U have won the £750 Pound prize. 2 claim is easy, call 087187272008 NOW1! Only 10p per minute. BT-national-rate."
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other suggestions?"
5570,ham,The guy did some bitching but I acted like i'd be interested in buying something else next week and he gave it to us for free


In [818]:
w2v_model = gensim.models.word2vec.Word2Vec.load("Word2Vec-local.bin")

In [819]:
le = LabelEncoder()
y = le.fit_transform(df['class']) 

In [820]:
y = pd.DataFrame(y)

In [821]:
y

,0
0,0
1,0
2,1
3,0
4,0
...,...
5567,1
5568,0
5569,0
5570,0


In [822]:
sentences = df['text']
lemmatizer = WordNetLemmatizer()
import re
corpus = []
for i in range(len(sentences)):
    review = re.sub('[^a-zA-Z]', ' ', sentences[i])
    review = review.lower()
    review = review.split()
    review = [lemmatizer.lemmatize(word) for word in review ]
    corpus.append(review)

In [823]:
vocab_size = 3570

In [824]:
max_input_length = 190

In [825]:
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(corpus)
sequences = tokenizer.texts_to_sequences(corpus)
padded = pad_sequences(sequences, maxlen=max_input_length)

In [826]:
X = padded
X

array([[   0,    0,    0, ...,   65,   63,  144],
       [   0,    0,    0, ...,  466,    6, 1747],
       [   0,    0,    0, ...,  398,  197,   17],
       ...,
       [   0,    0,    0, ...,   26,  112,  251],
       [   0,    0,    0, ...,    6,   13,   51],
       [   0,    0,    0, ...,    2,    7,  242]], dtype=int32)

In [827]:
X_train, X_test, y_train, y_test = train_test_split(padded, y, test_size=0.2, random_state=42)

In [828]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout,Bidirectional
from tensorflow.keras.optimizers import Adam

# ... (load your data, Word2Vec model, tokenizer, and hyperparameters like vocab_size, max_input_length) ...

model = Sequential()

# Embedding layer
model.add(Embedding(vocab_size, 100, input_length=max_input_length, 
                    weights=[w2v_model.wv.vectors]))
model.add(Bidirectional(LSTM(128, return_sequences=True)))
model.add(LSTM(128))
model.add(Dropout(0.2))
model.add(Dense(128))
model.add(Dense(64))
# Output layer with Softmax 
model.add(Dense(2, activation='softmax'))  

# Compile the model
optimizer = Adam(learning_rate=0.01)
model.compile(optimizer=optimizer, 
              loss='categorical_crossentropy',  # Use categorical cross-entropy with Softmax
              metrics=['accuracy'])

model.summary()

/Users/sanju/PycharmProjects/NLP_basics/.venv/lib/python3.9/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_42"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_42 (Embedding)        │ ?                      │       357,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_9 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_63 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_41 (Dropout)            │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_71 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_72 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_73 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 357,000 (1.36 MB)

 Trainable params: 357,000 (1.36 MB)

 Non-trainable params: 0 (0.00 B)

In [829]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((4457, 190), (1115, 190), (4457, 1), (1115, 1))

In [830]:
y_train

,0
1978,1
3989,0
3935,0
4078,0
4086,1
...,...
3772,0
5191,0
5226,0
5390,0


In [831]:
y_train = to_categorical(y_train, num_classes=2)  # Assuming 2 classes (spam, ham)
y_test = to_categorical(y_test, num_classes=2)


In [832]:
y_train

array([[0., 1.],
       [1., 0.],
       [1., 0.],
       ...,
       [1., 0.],
       [1., 0.],
       [1., 0.]])

In [833]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((4457, 190), (1115, 190), (4457, 2), (1115, 2))

In [834]:
model.fit(X_train, y_train, epochs=2, validation_data=(X_test, y_test))

Epoch 1/2
140/140 ━━━━━━━━━━━━━━━━━━━━ 48s 335ms/step - accuracy: 0.8355 - loss: 0.7406 - val_accuracy: 0.9121 - val_loss: 0.2631
Epoch 2/2
140/140 ━━━━━━━━━━━━━━━━━━━━ 67s 477ms/step - accuracy: 0.9291 - loss: 0.1823 - val_accuracy: 0.9812 - val_loss: 0.0666


In [835]:
def predict_class(text):
    """Predicts the class (spam or ham) for a new message."""
    processed_text = text
    sequence = tokenizer.texts_to_sequences([processed_text])
    padded_sequence = pad_sequences(sequence, maxlen=max_input_length)
    prediction = model.predict(padded_sequence)
    print(prediction)
    predicted_class = le.inverse_transform(prediction.argmax(axis=1))[0]
    return predicted_class


In [841]:
# Get user input
user_input = "Free entry "

# Predict class
predicted_class = predict_class(user_input)
print("Predicted class:", predicted_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
[[0.43661332 0.5633867 ]]
Predicted class: spam


In [837]:
l_q = []
l_i = []
for i in range(len(df['text'])):
    input_text = df['text'][i]
    predicted_class = predict_class(input_text)
    print("Predicted class:", predicted_class)
    if predicted_class == 'spam':
        l_q.append(input_text)
        l_i.append(i)
        break
        

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
[[0.9956986  0.00430146]]
Predicted class: ham
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
[[9.9994230e-01 5.7709134e-05]]
Predicted class: ham
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
[[0.00807867 0.9919213 ]]
Predicted class: spam


In [838]:
l_i

[2]

In [842]:
model.save('Spam_Classifier.keras')

In [843]:
model.save('Spam_Classifier.h5')